# The precision study (8 to 18 August 2026): every version, re-scored

Companion to `logs/PRECISION_ABLATION_LOG.md` (the short note) and `logs/ABLATION_LOG_V2.md` (the full log of the
study). It re-scores the stored runs of every version of the second prompt study with the strict scorer
(`src/eval_assets.py`, metric M-2), reads each arm's pre-registered rule and recall guard against the numbers, and
reproduces the measurements the study made before running (the concept oracle, the blind cap).

- **Runs:** `runs/assets_tuning18_<version>_r<k>/`, one JSON per module, written by `experiments/01_prompt_baselines/notebooks/finetuning_assetgen_v2.ipynb`
  in August. Kept locally; the public repository does not publish them, so a fresh clone scores nothing until they
  are restored.
- **Reference:** the LAsset authors' manual list (`data/ground_truth/manual_gt_neorv32.json`), 15 tuning modules,
  111 entries. Scored on the modules common to every run.
- **No API call** unless `RUN_API` is set in the first cell; then the last section regenerates one version with
  gpt-5-mini into a new run folder, as the August notebook did.
- Precision = hits / listed elements. Recall = hits / reference entries. "Paired" = the per-module difference between an
  arm and its baseline, modules as the pairing unit, with a 95% bootstrap interval. An interval that includes zero is
  a null at this sample size (15 modules). The **guard**: an arm whose mean recall is more than 0.03 below its
  baseline's is not promoted; 0.03 to 0.06 means re-run at five runs; more than 0.06 rejects.

In [1]:
RUN_API = False      # True: section 8 regenerates VERSION_TO_RUN with the OpenAI API. Nothing else calls an API.

import sys, os, io, json, hashlib, contextlib, re, time
from pathlib import Path
from collections import Counter
import pandas as pd

def _repo_root():
    for q in [Path.cwd(), *Path.cwd().parents]:
        if (q / "FINAL_NOTEBOOK.ipynb").exists():
            return q
    raise SystemExit("start this notebook inside the repository")

ROOT = _repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
import eval_assets as EV, diagnose_step1 as DG, rtl_parse
import prompts_v2 as P2                      # audit() runs at import: no identifier from the 41 modules in any core
import icl_examples_01b_noparse as ICL_NP, icl_examples_01b_ot as ICL_OT

pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 80)
REFS = EV.load_refs()
GT = REFS["gt"]
CLOSED = EV.load_closed()
print(f"repo root: {ROOT}")
print(f"manual reference: {len(GT)} modules, {sum(len(v) for v in GT.values())} entries (all 41 modules)")

repo root: E:\jobs\ff\test\Prepoison_subset
manual reference: 41 modules, 300 entries (all 41 modules)


## 1. The versions, and the prompt each run was made with

Every version is `core + "\n\n" + examples`. The cores are built in `src/prompts_v2.py` from one flat baseline by one
reversible edit per arm (a round-trip assert in each), so "identical outside the edit" holds by construction. The cell
recomposes each prompt and compares its SHA-256 with the one the August run loop stamped into `_run_meta.json`. For
the two arms that put the parsed list back, the run loop also stamped a fingerprint of the parse it read; that is
checked against the parse folders on disk.

In [2]:
NP, OT = ICL_NP.ICL_ASSET_EXAMPLES_01B_NOPARSE, ICL_OT.ICL_ASSET_EXAMPLES_01B_OT
PROMPT_VERSIONS = {               # version: (core, examples block)
    "v2":     (P2.ASSET_V2_BASE, NP),
    "v2s1":   (P2.ASSET_V2_S1,   NP),
    "v2r8":   (P2.ASSET_V2_R8,   NP),
    "v2x3":   (P2.ASSET_V2_BASE, OT),
    "v2x3r8": (P2.ASSET_V2_R8,   OT),
    "v2s3":   (P2.ASSET_V2_S3,   NP),
    "v2sec":  (P2.ASSET_V2_SEC,  NP),
    "v2rp":   (P2.ASSET_V2_RP,   NP),
    "v2p3":   (P2.ASSET_V2_P3,   NP),
    "v2p3c1": (P2.ASSET_V2_P3C1, NP),
}
# which parsed list a version receives in the user message (absent = RTL only)
PARSE_DIR = {"v2rp": "parsed_roles_tuning18", "v2p3": "parsed_v3_tuning18", "v2p3c1": "parsed_v3_tuning18"}

# arm, version, designated baseline, the one change, runs planned, registration
ARMS = [
    ("base",   "v2",     None, "v01c6p1p2 core + corrections D (closed-set parity), L (de-leak), N (neutralise); RTL only", 3, "registered"),
    ("S-1",    "v2s1",   "v2", "RULES: a container is an asset only when what it holds is itself an asset",            3, "registered"),
    ("R-8",    "v2r8",   "v2", "core: the A-08 close-each-concept pass removed",                                        5, "registered"),
    ("X-3",    "v2x3",   "v2", "examples: a third case study (OpenTitan gpio) with worked rejections",                  3, "NOT pre-registered"),
    ("X-3R-8", "v2x3r8", "v2", "R-8 core + X-3 examples",                                                              3, "NOT pre-registered"),
    ("S-3",    "v2s3",   "v2", "RULES: routed traffic belongs to the fabric; name the decision elements instead",      3, "registered"),
    ("SEC",    "v2sec",  "v2", "STEP 3: SAIF's demotion test; a secondary array per primary",                          3, "registered"),
    ("RP",     "v2rp",   "v2", "input: the parsed list back, with a role from a 152-label taxonomy, a relationship, an evidence line", 3, "registered"),
    ("P3",     "v2p3",   "v2", "input: the open-vocabulary v3 parse instead of the taxonomy (built, never run)",        0, "registered"),
    ("C1",     "v2p3c1", "v2", "P3 + a READING THE ANNOTATIONS block before STEP 1",                                    3, "registered"),
]
BASELINE = {v: b for _a, v, b, _w, _n, _r in ARMS if b}
VERSIONS = [v for _a, v, *_ in ARMS]

def composed(v):
    core, icl = PROMPT_VERSIONS[v]
    return core + "\n\n" + icl

rows = []
for arm, v, base, what, n, reg in ARMS:
    text = composed(v)
    sha = hashlib.sha256(text.encode()).hexdigest()
    dirs = sorted((ROOT / "runs").glob(f"assets_tuning18_{v}_r*"))
    stamped = {json.loads((d / "_run_meta.json").read_text())["system_prompt_sha256"] for d in dirs if (d / "_run_meta.json").exists()}
    fps = {json.loads((d / "_run_meta.json").read_text()).get("parse_fingerprint") for d in dirs if (d / "_run_meta.json").exists()} - {None, ""}
    rows.append({"arm": arm, "version": v, "baseline": base, "registration": reg, "run folders": len(dirs),
                 "core chars": len(PROMPT_VERSIONS[v][0]), "composed chars": len(text), "sha12 (src)": sha[:12],
                 "sha12 (run meta)": ",".join(sorted(s[:12] for s in stamped)) or "no runs",
                 "match": (stamped == {sha}) if stamped else None,
                 "parse fingerprint (run meta)": ",".join(sorted(f[:12] for f in fps)) or ""})
CHAIN = pd.DataFrame(rows)
display(CHAIN)
assert all(m for m in CHAIN["match"] if m is not None), "a run folder was made with a prompt that src/ no longer reproduces"

# the parse the C1 run read, against the three v3 parse folders on disk
import parse_v3
for d in ("parsed_v3_tuning18", "parsed_v3_tuning18_gen2", "parsed_v3_tuning18_prefix_gates"):
    if (ROOT / d).exists():
        print(f"{d:36s} fingerprint {parse_v3.parse_fingerprint(d)[:12]}")
print("(the v2p3c1 run meta names the first; the full log's parser table calls that content gen3, its C1 result line says gen2)")

,arm,version,baseline,registration,run folders,core chars,composed chars,sha12 (src),sha12 (run meta),match,parse fingerprint (run meta)
0,base,v2,None,registered,3,7412,70262,ea0608b316dc,ea0608b316dc,True,
1,S-1,v2s1,v2,registered,3,8117,70967,4fcdac76cce3,4fcdac76cce3,True,
2,R-8,v2r8,v2,registered,5,6862,69712,e02218e4c95d,e02218e4c95d,True,
3,X-3,v2x3,v2,NOT pre-registered,3,7412,111090,bdd9bb375f12,bdd9bb375f12,True,
4,X-3R-8,v2x3r8,v2,NOT pre-registered,3,6862,110540,3c543758f459,3c543758f459,True,
5,S-3,v2s3,v2,registered,3,8851,71701,6bd762261967,6bd762261967,True,
6,SEC,v2sec,v2,registered,3,9783,72633,1b0826ca22cf,1b0826ca22cf,True,
7,RP,v2rp,v2,registered,3,8066,70916,c18f0d40c62c,c18f0d40c62c,True,
8,P3,v2p3,v2,registered,0,8372,71222,ed45ec668d37,no runs,None,
9,C1,v2p3c1,v2,registered,3,11885,74735,ef4151232e86,ef4151232e86,True,540765753f16


parsed_v3_tuning18                   fingerprint 540765753f16
parsed_v3_tuning18_gen2              fingerprint 450d8de774ca
parsed_v3_tuning18_prefix_gates      fingerprint 5274280e4c2f
(the v2p3c1 run meta names the first; the full log's parser table calls that content gen3, its C1 result line says gen2)


## 2. What the model was given

The system prompt is the core (INPUTS, DEFINITIONS, METHOD with STEP 1 and STEP 2, RULES, OUTPUT CONTRACT) followed
by the worked examples. The user message is the comment-stripped RTL alone for every version except RP, P3 and C1,
which put the parsed ports and signals back, each element carrying the annotation its parser wrote. The cell builds
the user message for one module in each regime, exactly as the August notebook did, and shows its size.

In [3]:
RTL_DIR, PARSED_DIR = ROOT / "data/RTL_data", ROOT / "data/parsed_tuning18"
_ALL_BLOCKS = {"summary": True, "parsed": True, "rtl": True, "function": True}
INPUT_BLOCKS = {v: {"summary": False, "parsed": v in PARSE_DIR} for v in PROMPT_VERSIONS}

def build_asset_user(stem, summary, parsed, rtl, blocks=None):
    '''The user message, as the August notebook built it (verbatim logic).'''
    b = {**_ALL_BLOCKS, **(blocks or {})}
    ports, signals = parsed["ports"], parsed["signals"]
    if not b["function"]:
        ports = [{k: v for k, v in e.items() if k != "function"} for e in ports]
        signals = [{k: v for k, v in e.items() if k != "function"} for e in signals]
    parts = [f"TARGET IP MODULE: {stem}\n\n"]
    if b["summary"]:
        parts.append(f"=== TECHNICAL SUMMARY ===\n{summary}\n\n")
    if b["parsed"]:
        parts.append(f"=== PARSED I/O PORTS (JSON) ===\n{json.dumps(ports, indent=1)}\n\n")
        parts.append(f"=== PARSED INTERNAL SIGNALS (JSON) ===\n{json.dumps(signals, indent=1)}\n\n")
    if b["rtl"]:
        parts.append(f"=== RTL ===\n{rtl}\n\n")
    parts.append(f"Identify the primary security assets for '{stem}' and return the JSON object per the contract.")
    return "".join(parts)

def load_rtl(stem):
    return rtl_parse.strip_comments((RTL_DIR / f"{stem}.vhd").read_text(encoding="utf-8", errors="ignore"), vhdl=True)

def load_parsed(stem, version):
    '''The element list the version's run loop handed to build_asset_user and to validation.'''
    d = ROOT / PARSE_DIR.get(version, "data/parsed_tuning18")
    rec = json.loads((d / f"{stem}.json").read_text(encoding="utf-8"))
    if version in ("v2p3", "v2p3c1"):                    # the August loop normalised the v3 parse on load
        closed = {e["name"] for e in rec["ports"] + rec["signals"]}
        rec = parse_v3.normalize_record(rec, closed, Counter())
    return rec

stem = "neorv32_wdt"
rtl = load_rtl(stem)
print(f"{stem}: RTL {len(rtl)} chars (comments stripped)")
for name, v in (("v2 .. v2sec (RTL only)", "v2"), ("v2rp (parsed + taxonomy roles)", "v2rp"), ("v2p3c1 (parsed, v3 annotation)", "v2p3c1")):
    parsed = load_parsed(stem, v)
    u = build_asset_user(stem, "", parsed, rtl, INPUT_BLOCKS[v])
    fields = sorted({k for e in parsed["ports"][:1] + parsed["signals"][:1] for k in e})
    print(f"  {name:34s} user message {len(u):6d} chars   per-element fields: {fields}")
core, icl = PROMPT_VERSIONS["v2"]
print(f"\nsystem prompt v2: core {len(core.split())} words, examples {len(icl.split())} words; X-3 examples {len(OT.split())} words")

scored = sorted(m for m in GT if (PARSED_DIR / f"{m}.json").exists() and (RTL_DIR / f"{m}.vhd").exists())
P15 = {m: json.loads((PARSED_DIR / f"{m}.json").read_text(encoding="utf-8")) for m in scored}
n_elems = sum(len(p["ports"]) + len(p["signals"]) for p in P15.values())
n_names = sum(len({e["name"] for e in p["ports"] + p["signals"]}) for p in P15.values())
n_portfield = sum(1 for p in P15.values() for e in p["ports"] if "." in e["name"])
n_refs = sum(len(GT[m]) for m in scored)
print(f"{len(scored)} scored modules: {n_elems} closed-set elements ({n_names} distinct names), {n_portfield} of them port fields; "
      f"{n_refs} reference entries, {sum(1 for m in scored for n, _o in GT[m] if '.' in n and n.split('.')[0] in CLOSED[m]['port'])} of them port fields")

neorv32_wdt: RTL 4799 chars (comments stripped)
  v2 .. v2sec (RTL only)             user message   4943 chars   per-element fields: ['dir', 'entity', 'function', 'kind', 'name', 'type']
  v2rp (parsed + taxonomy roles)     user message  18591 chars   per-element fields: ['dir', 'entity', 'evidence', 'function', 'kind', 'name', 'relationship', 'roles', 'type']
  v2p3c1 (parsed, v3 annotation)     user message  30709 chars   per-element fields: ['dir', 'entity', 'evidence', 'function', 'functionality', 'kind', 'name', 'relationship', 'role', 'type']

system prompt v2: core 1143 words, examples 8018 words; X-3 examples 12014 words
15 scored modules: 1641 closed-set elements (1572 distinct names), 959 of them port fields; 111 reference entries, 0 of them port fields


## 3. Results: every version, mean over its runs

`ablate()` prints one row per version (mean over runs, with the sample sd of recall and F1), then recall by element
class and false positives by class. The two `LAsset` rows are the paper's own initial lists scored on the same modules:
the spec + RTL list is the one the August log quoted; the RTL-only list is the like-for-like comparison, since every
version here reads RTL only. The pooled table after it counts hits over all runs of a version.

In [4]:
RUNS = EV.collect(VERSIONS)            # warns for v2p3: built, never run
MODS = EV.common_modules(RUNS, GT)
paper_here = {m: v for m, v in REFS["paper"].items() if m in MODS}

def lasset_rtl_only():
    '''LAsset's published RTL-only initial list, in run shape, on the scored modules.'''
    d = json.loads((ROOT / "data/LAsset_initial_results/asset_list_neorv32_initial.json").read_text(encoding="utf-8"))
    out = {}
    for ip in d:
        if ip["IP"] in MODS:
            out[ip["IP"]] = [(a.get("Entity", ""), a["Asset RTL"], a.get("Security Objective", "")) for a in ip.get("Assets", []) if a.get("Asset RTL")]
    return out
rtl_only_here = lasset_rtl_only()
print(f"scored modules: {len(MODS)}, reference entries: {sum(len(GT[m]) for m in MODS)}; LAsset RTL-only list covers {len(rtl_only_here)} of them\n")
SCORES = EV.ablate(RUNS, GT, extra={"LAsset s+RTL": paper_here, "LAsset RTL": rtl_only_here},
                   title="precision study, every version, mean over runs (strict scorer, M-2)")

  [warn] no run directory for version 'v2p3'


scored modules: 15, reference entries: 111; LAsset RTL-only list covers 15 of them

precision study, every version, mean over runs (strict scorer, M-2)
scored on 15 modules common to every run
version       n  emit   TP   FP   FN      P  recall     sd      F1     sd  near
-------------------------------------------------------------------------------
v2            3   413  103  310    8  0.251   0.925  0.021   0.394  0.032   0.0
v2s1          3   425  104  321    7  0.246   0.937  0.009   0.389  0.024   0.0
v2r8          5   375   99  275   12  0.265   0.895  0.023   0.409  0.012   0.0
v2x3          3   391  100  291   11  0.259   0.901  0.018   0.402  0.048   0.0
v2x3r8        3   320   95  225   16  0.296   0.853  0.055   0.439  0.015   0.0
v2s3          3   420  105  316    6  0.249   0.943  0.014   0.394  0.009   0.0


v2sec         3   246   70  177   41  0.287   0.628  0.023   0.392  0.030   0.0
v2rp          3   449   95  354   16  0.213   0.856  0.000   0.340  0.023   0.3
v2p3c1        3   393   94  299   17  0.240   0.847  0.009   0.373  0.017   0.3
LAsset s+RTL  1   137  101   36   10  0.737   0.910          0.815         0.0
LAsset RTL    1   122   83   39   28  0.680   0.748          0.712         0.0

  recall by element class                   port       signal signal-field
  v2           (ref 47/32/31)         0.936        0.948        0.914     
  v2s1         (ref 47/32/31)         0.929        0.969        0.946     
  v2r8         (ref 47/32/31)         0.902        0.906        0.903     
  v2x3         (ref 47/32/31)         0.915        0.927        0.882     
  v2x3r8       (ref 47/32/31)         0.837        0.875        0.882     
  v2s3         (ref 47/32/31)         0.950        0.969        0.935     
  v2sec        (ref 47/32/31)         0.688        0.562        0.624     
 

In [5]:
rows = []
for arm, v, base, what, _n, reg in ARMS:
    if v not in SCORES:
        rows.append({"arm": arm, "version": v, "runs": 0, "what changed": what}); continue
    rs = SCORES[v]
    T, Fp, Fn = (sum(r[k] for r in rs) for k in ("tp", "fp", "fn"))
    rows.append({"arm": arm, "version": v, "runs": len(rs), "what changed": what,
                 "per run TP/FP/FN": "; ".join(f"{r['tp']}/{r['fp']}/{r['fn']}" for r in rs),
                 "pooled P": round(T / (T + Fp), 3), "pooled R": round(T / (T + Fn), 3),
                 "mean listed": round((T + Fp) / len(rs), 1), "mean P": round(EV._mean(r["precision"] for r in rs), 3),
                 "mean R": round(EV._mean(r["recall"] for r in rs), 3), "sd R": round(EV._sd(r["recall"] for r in rs), 3)})
POOLED = pd.DataFrame(rows)
display(POOLED)

,arm,version,runs,what changed,per run TP/FP/FN,pooled P,pooled R,mean listed,mean P,mean R,sd R
0,base,v2,3,"v01c6p1p2 core + corrections D (closed-set parity), L (de-leak), N (neutrali...",104/293/7; 100/270/11; 104/367/7,0.249,0.925,412.7,0.251,0.925,0.021
1,S-1,v2s1,3,RULES: a container is an asset only when what it holds is itself an asset,105/351/6; 104/285/7; 103/327/8,0.245,0.937,425.0,0.246,0.937,0.009
2,R-8,v2r8,5,core: the A-08 close-each-concept pass removed,97/267/14; 103/270/8; 99/267/12; 101/289/10; 97/283/14,0.265,0.895,374.6,0.265,0.895,0.023
3,X-3,v2x3,3,examples: a third case study (OpenTitan gpio) with worked rejections,100/248/11; 102/270/9; 98/356/13,0.256,0.901,391.3,0.259,0.901,0.018
4,X-3R-8,v2x3r8,3,R-8 core + X-3 examples,88/216/23; 100/230/11; 96/229/15,0.296,0.853,319.7,0.296,0.853,0.055
5,S-3,v2s3,3,RULES: routed traffic belongs to the fabric; name the decision elements instead,106/326/5; 103/318/8; 105/303/6,0.249,0.943,420.3,0.249,0.943,0.014
6,SEC,v2sec,3,STEP 3: SAIF's demotion test; a secondary array per primary,67/145/44; 70/166/41; 72/219/39,0.283,0.628,246.3,0.287,0.628,0.023
7,RP,v2rp,3,"input: the parsed list back, with a role from a 152-label taxonomy, a relati...",95/399/16; 95/326/16; 95/337/16,0.212,0.856,449.0,0.213,0.856,0.000
8,P3,v2p3,0,"input: the open-vocabulary v3 parse instead of the taxonomy (built, never run)",NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,C1,v2p3c1,3,P3 + a READING THE ANNOTATIONS block before STEP 1,95/330/16; 94/283/17; 93/285/18,0.239,0.847,393.3,0.240,0.847,0.009


## 4. Each arm against its baseline, its guard, and its pre-registered rule

Paired deltas (modules as the pairing unit; 95% bootstrap interval, 10,000 draws, seed 0) on precision, recall, F1 and
the three class recalls. The guard is read on the mean recall: drop ≤ 0.03 passes, 0.03 to 0.06 is indeterminate
(re-run at five runs), more than 0.06 fails. The hard checks are the named elements each arm had to keep; a group
fails when more of its entry-runs are missing than under `v2`. The rule column is the pass rule as the full log
registered it; the verdict is read off these numbers by code; the decision column is what the log did. X-3 and X-3R-8
had no registered rule and are judged against the standing guard only.

R-8 is read at three runs (the screen) and at five (the promotion test, matched against `v2`'s three).

In [6]:
def quiet(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*a, **k)

def fmt(r):
    return f"{r['delta']:+.3f} [{r['lo']:+.3f}, {r['hi']:+.3f}]"

def clears(r):
    return not (r["lo"] <= 0 <= r["hi"])

def first_k(runs, k):
    return {v: rs[:k] for v, rs in runs.items()}

def means(runs, v):
    rs = [EV.score(r, GT, True, only=MODS) for r in runs[v]]
    return {"P": EV._mean(r["precision"] for r in rs), "R": EV._mean(r["recall"] for r in rs),
            "emit": EV._mean(r["emit"] for r in rs), "n": len(rs)}

def deltas(runs, base, arm):
    out = {k: quiet(EV.paired, runs, base, arm, GT, key=k) for k in ("precision", "recall", "f1")}
    for c in ("port", "signal", "signal-field"):
        out[c] = quiet(EV.paired, runs, base, arm, GT, cls=c, closed=CLOSED)
    return out

def guard(drop):
    return "passes" if drop <= 0.03 else ("indeterminate: re-run at n=5" if drop <= 0.06 else "FAILS")

# hard checks: (module, element); the reference lists `bus` `state` twice, counted once here
CTRL17 = [("neorv32_wdt", "ctrl.enable"), ("neorv32_wdt", "ctrl.lock"), ("neorv32_wdt", "ctrl.timeout"),
          ("neorv32_uart", "ctrl.enable"), ("neorv32_uart", "ctrl.baud"), ("neorv32_uart", "ctrl.prsc"),
          ("neorv32_spi", "ctrl.enable"), ("neorv32_spi", "ctrl.cdiv"), ("neorv32_spi", "ctrl.prsc"),
          ("neorv32_twi", "ctrl.enable"), ("neorv32_twi", "ctrl.cdiv"), ("neorv32_twi", "ctrl.prsc"),
          ("neorv32_cpu_cp_muldiv", "ctrl.cnt"), ("neorv32_cpu_cp_muldiv", "ctrl.state"),
          ("neorv32_cpu_cp_muldiv", "ctrl.rs1_is_signed"), ("neorv32_cpu_cp_muldiv", "ctrl.rs2_is_signed"),
          ("neorv32_cache", "ctrl.buf_sync")]
BOUNDARY = [("neorv32_uart", "uart_rxd_i"), ("neorv32_uart", "uart_txd_o"), ("neorv32_spi", "spi_dat_i"), ("neorv32_twi", "twi_sda_i")]
BUS_DECISIONS = [("neorv32_bus", "sel"), ("neorv32_bus", "stb"), ("neorv32_bus", "port_sel"), ("neorv32_bus", "state")]
FSM = [("neorv32_bus", "state"), ("neorv32_bus", "sel"), ("neorv32_bus", "stb"), ("neorv32_cpu_cp_muldiv", "ctrl.state"),
       ("neorv32_hwspinlock", "sel"), ("neorv32_wdt", "cnt")]
HARD = {
    "v2s1":  {"buffer that is an asset + key_mem": [("neorv32_trng", "fifo.avail"), ("neorv32_trng", "fifo.free"), ("neorv32_trng", "fifo.re"), ("neorv32_cpu_cp_cfu", "key_mem")]},
    "v2s3":  {"traffic leaving the design": BOUNDARY, "decisions in the bus module": BUS_DECISIONS, "control record (17)": CTRL17},
    "v2sec": {"control record (17)": CTRL17, "crosses the design boundary": BOUNDARY, "FSM / arbitration state": FSM},
}

def emitted_runs(runs, v, mod, name):
    return sum(1 for r in runs[v] if any(n == name for _e, n, _o in r.get(mod, [])))

def missing(runs, v, group):
    return sum(len(runs[v]) - emitted_runs(runs, v, m, n) for m, n in group), len(runs[v]) * len(group)

def hard_checks(runs, v, base):
    '''-> (all hold, text). A group fails when more entry-runs are missing than under the baseline.'''
    if v not in HARD:
        return None, ""
    ok, parts = True, []
    for label, group in HARD[v].items():
        ma, ta = missing(runs, v, group); mb, tb = missing(runs, base, group)
        held = ma <= mb
        ok &= held
        parts.append(f"{label}: missing {ma}/{ta} (base {mb}/{tb}){'' if held else ' FAIL'}")
    return ok, "; ".join(parts)

# the registered rules, as code: each returns (verdict, one-line reading)
def rule_none(d, a, b, g, hc):
    return "first run of the clean base", f"P {a['P']:.3f} R {a['R']:.3f}"
def rule_S1(d, a, b, g, hc):
    p_up = clears(d["precision"]) and d["precision"]["delta"] > 0
    if p_up and g == "passes" and hc[0]:
        return "PASS: promote at n=5", "precision up, guard passes, hard checks hold"
    return ("NULL: record, do not re-word" if not clears(d["precision"]) else "FAIL"), f"precision {fmt(d['precision'])}; guard {g}"
def rule_R8(d, a, b, g, hc):
    p_up = clears(d["precision"]) and d["precision"]["delta"] > 0
    if not p_up:
        return "NULL: not promoted", f"precision {fmt(d['precision'])}; guard {g}"
    return ("PASS: promote at n=5" if g == "passes" else ("RE-RUN at n=5" if g.startswith("indet") else "FAIL: the sweep stays")), f"precision {fmt(d['precision'])}; guard {g}"
def rule_guard_only(d, a, b, g, hc):
    p_up = clears(d["precision"]) and d["precision"]["delta"] > 0
    return (f"guard {g}; precision {'up' if p_up else 'null'}"), "no registered rule: standing guard only"
def rule_S3(d, a, b, g, hc):
    p_up = clears(d["precision"]) and d["precision"]["delta"] > 0
    if not hc[0]:
        return "FAIL: a hard check failed", f"precision {fmt(d['precision'])}; guard {g}"
    if p_up and g == "passes":
        return "PASS: promote at n=5", f"precision {fmt(d['precision'])}; guard {g}"
    return ("NULL: record, do not re-word" if not clears(d["precision"]) else "FAIL: precision moved the wrong way"), f"precision {fmt(d['precision'])}; guard {g}"
def rule_SEC(d, a, b, g, hc):
    moved = SEC_MOVE["left"] > 0 and SEC_MOVE["back"] / SEC_MOVE["left"] >= 0.5
    if not moved:
        return "REJECT: the list was added, not moved", f"{SEC_MOVE['back']} of {SEC_MOVE['left']} departures reappear as secondary"
    p_up = clears(d["precision"]) and d["precision"]["delta"] > 0
    if p_up and g == "passes" and hc[0]:
        return "PASS: promote at n=5", "mechanism check passed"
    if g.startswith("indet") and hc[0]:
        return "RE-RUN at n=5", "mechanism check passed"
    return "FAIL", f"mechanism check passed ({SEC_MOVE['back']}/{SEC_MOVE['left']} moved); precision {fmt(d['precision'])}; guard {g}"
def rule_RP(d, a, b, g, hc):
    flat_r = abs(a["R"] - b["R"]) <= 0.03
    flat_e = abs(a["emit"] - b["emit"]) <= 0.10 * b["emit"]
    if flat_r and flat_e:
        return "FLAT: the roles carry the coverage; run R-8 on this base next", f"recall {a['R']:.3f} vs {b['R']:.3f}, listed {a['emit']:.0f} vs {b['emit']:.0f}"
    if a["R"] > b["R"] and a["emit"] > b["emit"]:
        return "UP: a second generator, not a substitute", f"recall {a['R']:.3f}, listed {a['emit']:.0f}"
    return "RECALL DOWN: the list displaces the reading of the RTL", f"recall {fmt(d['recall'])}; port recall {fmt(d['port'])}"
def rule_C1(d, a, b, g, hc):
    flat_r = abs(a["R"] - b["R"]) <= 0.03
    if a["emit"] < b["emit"] - 0.10 * b["emit"] and flat_r:
        return "listed down, recall flat: the reading procedure was what was missing", f"listed {a['emit']:.0f} vs {b['emit']:.0f}"
    if a["emit"] > b["emit"]:
        return "listed up: the list is a generator", f"listed {a['emit']:.0f} vs {b['emit']:.0f}"
    return f"listed {a['emit']:.0f} vs {b['emit']:.0f}, recall {'flat' if flat_r else 'down'}; guard {g}", f"precision {fmt(d['precision'])}; recall {fmt(d['recall'])}"

RULES = {"v2": rule_none, "v2s1": rule_S1, "v2r8": rule_R8, "v2x3": rule_guard_only, "v2x3r8": rule_guard_only,
         "v2s3": rule_S3, "v2sec": rule_SEC, "v2rp": rule_RP, "v2p3c1": rule_C1}
DECISION = {   # what the full log did
    "v2": "base", "v2s1": "rejected: null; the carve-out fired everywhere in the TRNG",
    "v2r8": "not promoted (precision null); mechanism confirmed; kept for the combination",
    "v2x3": "not promoted: null; same dial as R-8", "v2x3r8": "rejected: guard fails; first distinguishable precision gain",
    "v2s3": "rejected: precision down; the redirect became a licence", "v2sec": "rejected: guard fails; the movement check passed",
    "v2rp": "rejected: recall down, the list displaces RTL reading", "v2p3c1": "rejected: no precision gain, recall unstable"}

# the SEC movement check, needed by rule_SEC: distinct primaries that left, and how many came back as secondary
def primaries(v, i):
    return {(m, x[1]) for m in MODS for x in RUNS[v][i][m]}
def secondaries(v, i):
    out = set()
    for m in MODS:
        d = json.loads((ROOT / "runs" / f"assets_tuning18_{v}_r{i}" / f"{m}.json").read_text(encoding="utf-8"))
        for a in d.get("Assets", []):
            out.update((m, s) for s in (a.get("secondary") or []))
    return out
_ua = set().union(*[primaries("v2", i) for i in range(3)])
_ub = set().union(*[primaries("v2sec", i) for i in range(3)])
_us = set().union(*[secondaries("v2sec", i) for i in range(3)])
SEC_MOVE = {"v2": len(_ua), "v2sec": len(_ub), "left": len(_ua - _ub), "back": len((_ua - _ub) & _us),
            "both per run": [len(primaries("v2sec", i) & secondaries("v2sec", i)) for i in range(3)]}

def arm_table(runs, arms, title):
    rows = []
    for arm, v, base, what, _n, reg in arms:
        if v not in runs or (base and base not in runs):
            continue
        a = means(runs, v)
        if base is None:
            rows.append({"arm": arm, "version": v, "n": a["n"], "P": round(a["P"], 3), "R": round(a["R"], 3), "listed": round(a["emit"], 1),
                         "verdict (code)": RULES[v](None, a, None, "", None)[0], "decision (log)": DECISION[v]}); continue
        b = means(runs, base)
        d = deltas(runs, base, v)
        drop = b["R"] - a["R"]
        g = guard(drop)
        hc = hard_checks(runs, v, base)
        verdict, reading = RULES[v](d, a, b, g, hc)
        rows.append({"arm": arm, "version": v, "n": a["n"], "P": round(a["P"], 3), "R": round(a["R"], 3), "listed": round(a["emit"], 1),
                     "dP": fmt(d["precision"]), "dR": fmt(d["recall"]), "dF1": fmt(d["f1"]),
                     "port R": fmt(d["port"]), "signal R": fmt(d["signal"]), "signal-field R": fmt(d["signal-field"]),
                     "drop": round(drop, 3), "guard": g, "hard checks": hc[1] if hc[0] is not None else "",
                     "verdict (code)": verdict, "reading": reading, "decision (log)": DECISION[v]})
    df = pd.DataFrame(rows)
    print(title)
    display(df.drop(columns=["reading", "hard checks"]))
    return df

SCREEN = arm_table(first_k(RUNS, 3), ARMS, "each arm at three runs, against three runs of its baseline (the screen)")
print("\nhard checks and readings:")
for _i, r in SCREEN.iterrows():
    hc, rd = (str(r.get("hard checks") or "") if pd.notna(r.get("hard checks")) else ""), (str(r.get("reading") or "") if pd.notna(r.get("reading")) else "")
    if hc or rd:
        print(f"  {r['arm']:7s} {rd}" + (f"\n          hard checks: {hc}" if hc else ""))
print(f"\nSEC movement: distinct primary elements {SEC_MOVE['v2']} -> {SEC_MOVE['v2sec']}; {SEC_MOVE['left']} left, "
      f"{SEC_MOVE['back']} reappear as secondary ({SEC_MOVE['back'] / SEC_MOVE['left']:.0%}); "
      f"elements both primary and secondary per run {SEC_MOVE['both per run']}")

each arm at three runs, against three runs of its baseline (the screen)


,arm,version,n,P,R,listed,verdict (code),decision (log),dP,dR,dF1,port R,signal R,signal-field R,drop,guard
0,base,v2,3,0.251,0.925,412.7,first run of the clean base,base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,S-1,v2s1,3,0.246,0.937,425.0,"NULL: record, do not re-word",rejected: null; the carve-out fired everywhere in the TRNG,"-0.009 [-0.023, +0.006]","+0.010 [-0.010, +0.034]","-0.006 [-0.021, +0.010]","+0.004 [-0.046, +0.074]","+0.009 [+0.000, +0.028]","+0.043 [+0.000, +0.086]",-0.012,passes
2,R-8,v2r8,3,0.271,0.898,367.7,NULL: not promoted,not promoted (precision null); mechanism confirmed; kept for the combination,"+0.005 [-0.011, +0.019]","-0.019 [-0.044, +0.009]","+0.007 [-0.011, +0.022]","-0.019 [-0.065, +0.019]","-0.025 [-0.050, -0.005]","-0.019 [-0.086, +0.043]",0.027,passes
3,X-3,v2x3,3,0.259,0.901,391.3,guard passes; precision null,not promoted: null; same dial as R-8,"-0.005 [-0.024, +0.012]","-0.016 [-0.049, +0.016]","-0.003 [-0.023, +0.017]","-0.028 [-0.093, +0.018]","-0.010 [-0.031, +0.000]","-0.031 [-0.148, +0.056]",0.024,passes
4,X-3R-8,v2x3r8,3,0.296,0.853,319.7,guard FAILS; precision up,rejected: guard fails; first distinguishable precision gain,"+0.025 [+0.013, +0.038]","-0.054 [-0.096, -0.016]","+0.023 [+0.008, +0.038]","-0.120 [-0.202, -0.043]","-0.035 [-0.083, +0.000]","-0.037 [-0.148, +0.037]",0.072,FAILS
5,S-3,v2s3,3,0.249,0.943,420.3,FAIL: a hard check failed,rejected: precision down; the redirect became a licence,"-0.016 [-0.036, -0.001]","+0.017 [+0.000, +0.039]","-0.015 [-0.033, +0.002]","+0.023 [-0.032, +0.093]","+0.009 [+0.000, +0.028]","+0.025 [+0.000, +0.062]",-0.018,passes
6,SEC,v2sec,3,0.287,0.628,246.3,FAIL,rejected: guard fails; the movement check passed,"+0.069 [+0.006, +0.131]","-0.300 [-0.379, -0.220]","-0.012 [-0.073, +0.046]","-0.312 [-0.435, -0.194]","-0.341 [-0.486, -0.178]","-0.201 [-0.373, -0.062]",0.297,FAILS
7,RP,v2rp,3,0.213,0.856,449.0,RECALL DOWN: the list displaces the reading of the RTL,"rejected: recall down, the list displaces RTL reading","-0.054 [-0.084, -0.028]","-0.055 [-0.100, -0.018]","-0.063 [-0.095, -0.036]","-0.115 [-0.215, -0.022]","-0.028 [-0.075, +0.000]","+0.031 [+0.000, +0.074]",0.069,FAILS
8,C1,v2p3c1,3,0.240,0.847,393.3,"listed 393 vs 413, recall down; guard FAILS","rejected: no precision gain, recall unstable","-0.038 [-0.071, -0.005]","-0.057 [-0.104, -0.011]","-0.044 [-0.081, -0.007]","-0.061 [-0.143, +0.005]","-0.078 [-0.167, -0.016]","-0.056 [-0.185, +0.056]",0.078,FAILS



hard checks and readings:
  S-1     precision -0.009 [-0.023, +0.006]; guard passes
          hard checks: buffer that is an asset + key_mem: missing 2/12 (base 3/12)
  R-8     precision +0.005 [-0.011, +0.019]; guard passes
  X-3     no registered rule: standing guard only
  X-3R-8  no registered rule: standing guard only
  S-3     precision -0.016 [-0.036, -0.001]; guard passes
          hard checks: traffic leaving the design: missing 0/12 (base 0/12); decisions in the bus module: missing 1/12 (base 0/12) FAIL; control record (17): missing 0/51 (base 1/51)
  SEC     mechanism check passed (117/123 moved); precision +0.069 [+0.006, +0.131]; guard FAILS
          hard checks: control record (17): missing 9/51 (base 1/51) FAIL; crosses the design boundary: missing 3/12 (base 0/12) FAIL; FSM / arbitration state: missing 10/18 (base 0/18) FAIL
  RP      recall -0.055 [-0.100, -0.018]; port recall -0.115 [-0.215, -0.022]
  C1      precision -0.038 [-0.071, -0.005]; recall -0.057 [-0.104,

In [7]:
# R-8 at its full five runs against v2's three (the promotion reading); the other arms have three runs only
FULL = arm_table(RUNS, [x for x in ARMS if x[1] in ("v2", "v2r8")], "R-8 at five runs (unmatched n, as the log read it for promotion)")

R-8 at five runs (unmatched n, as the log read it for promotion)


,arm,version,n,P,R,listed,verdict (code),decision (log),dP,dR,dF1,port R,signal R,signal-field R,drop,guard
0,base,v2,3,0.251,0.925,412.7,first run of the clean base,base,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,R-8,v2r8,5,0.265,0.895,374.6,NULL: not promoted,not promoted (precision null); mechanism confirmed; kept for the combination,"-0.000 [-0.014, +0.012]","-0.024 [-0.043, -0.007]","-0.000 [-0.014, +0.013]","-0.035 [-0.083, +0.011]","-0.023 [-0.046, -0.005]","-0.006 [-0.067, +0.053]",0.029,passes


## 5. Measured before running: the concept oracle and the blind cap

Two readings the study took from `v01c6p1p2`'s stored output before spending a run (full log, sections 6 and 8).
The oracle deletes every concept that produced no hit, the most a concept-level rule could ever do; the cap keeps
only the first *k* elements the model wrote under each concept, in its own order. Both use the concept id each
listed element carries.

In [8]:
def tp_indices(preds, ref):
    '''Indices of the listed elements the scorer credits (same matching as EV.score).'''
    used = set()
    for rname, _o in ref:
        cand = {i: preds[i][1] for i in range(len(preds)) if i not in used}
        pi = EV._hit_idx(cand, rname, True)
        if pi is not None:
            used.add(pi)
    return used

def concept_runs(v):
    return [DG.load_concept_runs(ROOT / "runs" / f"assets_tuning18_{v}_r{i}") for i in range(3)]

def oracle_delete(v):
    res = []
    for cr in concept_runs(v):
        run = {}
        for m in MODS:
            assets = cr[m]["assets"]
            hits = tp_indices([(e, n, o) for e, n, o, _c in assets], GT[m])
            good = {assets[i][3] for i in hits}
            run[m] = [(e, n, o) for e, n, o, c in assets if c in good]
        res.append(EV.score(run, GT, True, only=MODS))
    return res

def cap_first(v, k):
    res = []
    for cr in concept_runs(v):
        run = {}
        for m in MODS:
            seen, out = Counter(), []
            for e, n, o, c in cr[m]["assets"]:
                seen[c] += 1
                if seen[c] <= k:
                    out.append((e, n, o))
            run[m] = out
        res.append(EV.score(run, GT, True, only=MODS))
    return res

def row(label, rs):
    return {"": label, "listed": round(EV._mean(r["emit"] for r in rs), 1), "TP": round(EV._mean(r["tp"] for r in rs), 1),
            "P": round(EV._mean(r["precision"] for r in rs), 3), "R": round(EV._mean(r["recall"] for r in rs), 3),
            "F1": round(EV._mean(r["f1"] for r in rs), 3)}

rows = []
for v in ("v01c6p1p2", "v2"):
    base = [EV.score(r, GT, True, only=MODS) for r in EV.collect([v])[v][:3]]
    rows.append(row(f"{v} as generated", base))
    rows.append(row(f"{v}, every concept with no hit deleted (oracle)", oracle_delete(v)))
    rows.append(row(f"{v}, every false positive deleted (perfect filter)", [{**r, "emit": r["tp"], "fp": 0, "precision": 1.0, "f1": 2 * r["recall"] / (1 + r["recall"])} for r in base]))
ORACLE = pd.DataFrame(rows); display(ORACLE)
o = ORACLE.iloc[1]; g = ORACLE.iloc[0]
print(f"v01c6p1p2: the oracle keeps every hit and reaches P {o['P']:.3f}; the false positives it cannot remove, "
      f"{(o['listed'] - o['TP']) / (g['listed'] - g['TP']):.0%} of all, sit inside concepts that also have a hit")

CAP = pd.DataFrame([row(f"cap {k if k < 99 else 'none'}", cap_first("v01c6p1p2", k)) for k in (99, 6, 4, 3, 1)])
display(CAP)
print("the model's own order carries no ranking: hits and false positives fall in the same proportion (R-V2-01, rejected before running)")

,,listed,TP,P,R,F1
0,v01c6p1p2 as generated,404.0,98.3,0.243,0.886,0.382
1,"v01c6p1p2, every concept with no hit deleted (oracle)",271.7,98.3,0.362,0.886,0.514
2,"v01c6p1p2, every false positive deleted (perfect filter)",98.3,98.3,1.000,0.886,0.939
3,v2 as generated,412.7,102.7,0.251,0.925,0.394
4,"v2, every concept with no hit deleted (oracle)",290.7,102.7,0.357,0.925,0.514
5,"v2, every false positive deleted (perfect filter)",102.7,102.7,1.000,0.925,0.961


v01c6p1p2: the oracle keeps every hit and reaches P 0.362; the false positives it cannot remove, 57% of all, sit inside concepts that also have a hit


,,listed,TP,P,R,F1
0,cap none,404.0,98.3,0.243,0.886,0.382
1,cap 6,350.7,89.3,0.255,0.805,0.387
2,cap 4,286.3,76.0,0.265,0.685,0.382
3,cap 3,237.0,67.3,0.284,0.607,0.387
4,cap 1,90.7,35.0,0.386,0.315,0.347


the model's own order carries no ranking: hits and false positives fall in the same proportion (R-V2-01, rejected before running)


## 6. Mechanism: the fan-out dial, and which class each arm cut

Elements bound per concept, the largest single concept, and concepts per module, from the `ConceptualAssets` array
and the concept id on each listed element. R-8, X-3 and their combination move this dial; S-1, S-3 and SEC were
meant not to. Then the listings per element class, with the change against `v2`: X-3R-8's cut fell hardest on ports,
the class with the best precision; S-3's redirect added signals and signal fields, the two worst.

In [9]:
rows = []
for v in ("v2", "v2s1", "v2r8", "v2x3", "v2x3r8", "v2s3", "v2sec"):
    ec, mx, cm = [], [], []
    for cr in concept_runs(v):
        nc = ne = 0; m_ = 0
        for m in MODS:
            rec = cr[m]; nc += len(rec["concepts"]); ne += len(rec["assets"])
            b = Counter(cid for *_x, cid in rec["assets"] if cid)
            if b:
                m_ = max(m_, max(b.values()))
        ec.append(ne / nc); mx.append(m_); cm.append(nc / len(MODS))
    rows.append({"version": v, "elements per concept": round(EV._mean(ec), 2), "largest concept (max over runs)": max(mx),
                 "concepts per module": round(EV._mean(cm), 2)})
FANOUT = pd.DataFrame(rows); display(FANOUT)

def class_listings(v):
    cs = [EV.by_class(r, GT, CLOSED) for r in SCORES[v][:3]]
    return {c: EV._mean(x[c]["tp"] + x[c]["fp"] for x in cs) for c in ("port", "signal", "signal-field")}
def class_precision(v):
    cs = [EV.by_class(r, GT, CLOSED) for r in SCORES[v][:3]]
    return {c: EV._mean(x[c]["tp"] / (x[c]["tp"] + x[c]["fp"]) for x in cs) for c in ("port", "signal", "signal-field")}
base = class_listings("v2"); bp = class_precision("v2")
print("v2 precision by class: " + ", ".join(f"{c} {bp[c]:.3f}" for c in bp) + "   (signal fields carry the most false positives at the lowest precision)")
rows = [{"version": "v2", **{f"{c} listed": f"{base[c]:.1f}" for c in base}}]
for v in ("v2r8", "v2x3", "v2x3r8", "v2s3", "v2sec"):
    L = class_listings(v)
    rows.append({"version": v, **{f"{c} listed": f"{L[c]:.1f} ({(L[c] - base[c]) / base[c]:+.0%})" for c in L}})
CLASSCUT = pd.DataFrame(rows); display(CLASSCUT)

,version,elements per concept,largest concept (max over runs),concepts per module
0,v2,4.47,53,6.16
1,v2s1,4.64,50,6.11
2,v2r8,3.76,12,6.53
3,v2x3,4.15,47,6.29
4,v2x3r8,3.54,20,6.04
5,v2s3,4.41,19,6.36
6,v2sec,2.69,45,6.16


v2 precision by class: port 0.300, signal 0.307, signal-field 0.181   (signal fields carry the most false positives at the lowest precision)


,version,port listed,signal listed,signal-field listed
0,v2,154.0,98.7,157.7
1,v2r8,125.7 (-18%),84.7 (-14%),154.3 (-2%)
2,v2x3,145.0 (-6%),98.3 (-0%),146.0 (-7%)
3,v2x3r8,105.7 (-31%),80.7 (-18%),131.0 (-17%)
4,v2s3,133.0 (-14%),110.3 (+12%),174.3 (+11%)
5,v2sec,117.3 (-24%),45.0 (-54%),82.0 (-48%)


## 7. The parsed-list arms: union against core recall, and names outside the closed set

Union = a reference entry found in at least one run; core = found in every run. All arms reach about the same union;
the parsed-list arms lose the core: the model's choice shifts run to run. The second table is from each run's
`_validation.json`: the names a version used that were not in the regex closed set. Correction D cut them 187 → 106
over three runs between `v01c6p1p2` and `v2`; with the list in the prompt there are none.

In [10]:
def union_or_core(rs, mode):
    out = {}
    for m in MODS:
        per = [Counter((e, n) for e, n, _o in r.get(m, [])) for r in rs]
        keys = set().union(*[set(p) for p in per])
        out[m] = []
        for k in keys:
            cs = [p.get(k, 0) for p in per]
            out[m] += [(k[0], k[1], "")] * (max(cs) if mode == "union" else min(cs))
    return out

rows = []
for v in ("v2", "v2r8", "v2x3", "v2x3r8", "v2rp", "v2p3c1"):
    rs = RUNS[v][:3]
    u = EV.score(union_or_core(rs, "union"), GT, True, only=MODS); c = EV.score(union_or_core(rs, "core"), GT, True, only=MODS)
    rows.append({"version": v, "mean R": round(EV._mean(SCORES[v][i]["recall"] for i in range(3)), 3),
                 "union R (3 runs)": round(u["recall"], 3), "union hits": u["tp"], "core R (in all 3)": round(c["recall"], 3), "core hits": c["tp"],
                 "gap": round(u["recall"] - c["recall"], 3)})
UNION = pd.DataFrame(rows); display(UNION)
print()
VAL = EV.validation_table(["v01c6p1p2", "v2", "v2s1", "v2r8", "v2x3", "v2x3r8", "v2s3", "v2sec", "v2rp", "v2p3c1"])

,version,mean R,union R (3 runs),union hits,core R (in all 3),core hits,gap
0,v2,0.925,0.955,106,0.874,97,0.081
1,v2r8,0.898,0.955,106,0.811,90,0.144
2,v2x3,0.901,0.955,106,0.784,87,0.171
3,v2x3r8,0.853,0.928,103,0.721,80,0.207
4,v2rp,0.856,0.946,105,0.748,83,0.198
5,v2p3c1,0.847,0.928,103,0.748,83,0.180



  [warn] assets_tuning18_v01c6p1p2_r3/_validation.json covers 0/18 modules -- INCOMPLETE, excluded
  [warn] assets_tuning18_v01c6p1p2_r4/_validation.json covers 0/18 modules -- INCOMPLETE, excluded


  version       n  assets  ungnd   rate |  in RTL  near<=2  invented |  ent.mism  bad obj  concept
  v01c6p1p2     3     493   62.3  12.6% |    62.3      0.0       0.0 |       0.0      0.0      0.0
  v2            3     475   35.3   7.4% |    35.3      0.0       0.0 |       0.0      0.0      0.0
  v2s1          3     496   43.7   8.8% |    43.7      0.0       0.0 |       8.7      0.0      0.0
  v2r8          5     434   33.4   7.7% |    33.4      0.0       0.0 |       0.0      0.0      0.0
  v2x3          3     448   30.0   6.7% |    30.0      0.0       0.0 |       8.0      0.0      0.0
  v2x3r8        3     372   27.7   7.4% |    27.0      0.0       0.7 |       9.7      0.0      0.7
  v2s3          3     481   34.0   7.1% |    33.7      0.0       0.3 |       1.0      0.0      0.0
  v2sec         3     298   33.3  11.2% |    33.3      0.0       0.0 |       9.7      0.0      0.0
  v2rp          3     479    0.0   0.0% |     0.0      0.0       0.0 |       0.0      0.0      0.7
  v2p3c1  

## 8. Regenerating a version (off by default)

With `RUN_API = True`, an `API.env` file holding `OPENAI_API_KEY=...` in the repository root, and the parse folders on
disk, this cell runs `VERSION_TO_RUN` for `REPEATS` runs with gpt-5-mini, as the August loop did: one call per module,
the response kept verbatim under `_raw/`, the prompt's SHA-256 (and, for a parsed-list version, the parse fingerprint)
stamped in `_run_meta.json`, a `_validation.json` per run, and a refusal to write into a folder made with a different
prompt or parse. Validation is report-only for the RTL-only versions; a version that receives the parsed list gets one
corrective regeneration when a listed name is not in it, as in August. One 3-run arm is 54 calls. The default is
`v2p3`, the version that was built and never run. The August runs are not reproduced exactly by rerunning: generation
is not deterministic, and the model behind the gpt-5-mini name may have changed.

In [11]:
VERSION_TO_RUN, REPEATS, EXEC_MODEL = "v2p3", 3, "gpt-5-mini"

def validate_primary(result, parsed, expect_concepts):
    '''One record per problem: a name not in the closed set, an entity that does not declare it, a bad objective,
    a concept-contract violation. The August loop also recorded whether an ungrounded name exists in the RTL.'''
    elems = parsed["ports"] + parsed["signals"]
    keys = {(e["entity"], e["name"]) for e in elems}
    names = {e["name"] for e in elems}
    out = []
    for a in result.get("Assets", []):
        r, en, obj = a.get("Asset RTL"), a.get("Entity"), a.get("Security Objective")
        if r not in names:
            out.append({"kind": "ungrounded", "name": r, "detail": f"ungrounded Asset RTL '{r}'"})
        elif (en, r) not in keys:
            out.append({"kind": "entity_mismatch", "name": r, "detail": f"Entity mismatch for '{r}': '{en}'"})
        if obj not in {"Confidentiality", "Integrity", "Availability"}:
            out.append({"kind": "bad_objective", "name": r, "detail": f"bad objective for '{r}': '{obj}'"})
    return out + [{"kind": "concept_contract", "name": None, "detail": s} for s in DG.validate_concepts(result, expect_concepts)]

def _loads(txt):
    try:
        return json.loads(txt)
    except Exception:
        m = re.search(r"\{.*\}", txt, re.S)
        return json.loads(m.group(0)) if m else {}

if RUN_API:
    from dotenv import load_dotenv
    from openai import OpenAI
    load_dotenv(ROOT / "API.env")
    oai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    system = composed(VERSION_TO_RUN)
    sha = hashlib.sha256(system.encode()).hexdigest()
    blocks = INPUT_BLOCKS[VERSION_TO_RUN]
    expect_concepts = "ConceptualAssets" in system
    retry_on_validation = blocks["parsed"]
    parse_fp = parse_v3.parse_fingerprint(PARSE_DIR[VERSION_TO_RUN]) if VERSION_TO_RUN in ("v2p3", "v2p3c1") else ""
    modules = sorted(p.stem for p in RTL_DIR.glob("*.vhd"))

    def call(stem, parsed_m, out_dir):
        user = build_asset_user(stem, "", parsed_m, load_rtl(stem), blocks)
        r = oai.responses.create(model=EXEC_MODEL, instructions=system, input=user, reasoning={"effort": "high"},
                                 max_output_tokens=65536, text={"format": {"type": "json_object"}})
        txt = r.output_text
        (out_dir / "_raw").mkdir(exist_ok=True)
        (out_dir / "_raw" / f"{stem}.json").write_text(txt, encoding="utf-8")
        obj = _loads(txt)
        if not (isinstance(obj, dict) and isinstance(obj.get("Assets"), list)):
            raise RuntimeError(f"{stem}: unparseable asset response, nothing cached")
        return {"IP": stem, **{k: v for k, v in obj.items() if k != "IP"}}

    for rep in range(REPEATS):
        out_dir = ROOT / "runs" / f"assets_tuning18_{VERSION_TO_RUN}_r{rep}"
        out_dir.mkdir(parents=True, exist_ok=True)
        meta_p = out_dir / "_run_meta.json"
        if meta_p.exists():
            old = json.loads(meta_p.read_text())
            if old["system_prompt_sha256"] != sha or (parse_fp and old.get("parse_fingerprint") not in (None, parse_fp)):
                raise RuntimeError(f"{out_dir} holds assets from another prompt or parse; use a new version name")
        else:
            meta_p.write_text(json.dumps({"version": VERSION_TO_RUN, "repeat": rep, "model": EXEC_MODEL, "effort": "high",
                                          "system_prompt_sha256": sha, "system_prompt_chars": len(system),
                                          "parse_fingerprint": parse_fp, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=2))
        report = {"version": VERSION_TO_RUN, "repeat": rep, "model": EXEC_MODEL, "system_prompt_sha256": sha,
                  "retry_on_validation": retry_on_validation, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), "modules": {}, "totals": {}}
        for stem in modules:
            parsed_m = load_parsed(stem, VERSION_TO_RUN)
            outp = out_dir / f"{stem}.json"
            if outp.exists():
                res = json.loads(outp.read_text(encoding="utf-8")); note = "[skip] on disk"
            else:
                res = call(stem, parsed_m, out_dir); note = "[ok  ]"
                recs = validate_primary(res, parsed_m, expect_concepts)
                if recs and retry_on_validation:                      # one corrective regeneration, keep the better
                    res2 = call(stem, parsed_m, out_dir)
                    if len(validate_primary(res2, parsed_m, expect_concepts)) <= len(recs):
                        res = res2
                outp.write_text(json.dumps(res, indent=2), encoding="utf-8")
            recs = validate_primary(res, parsed_m, expect_concepts)
            report["modules"][stem] = {"assets": len(res.get("Assets", [])), "concepts": len(res.get("ConceptualAssets", []) or []),
                                       "by_kind": dict(Counter(r["kind"] for r in recs)), "issues": recs}
            print(f"{note} r{rep} {stem:24s} {len(res['Assets']):3d} assets" + (f"   {dict(Counter(r['kind'] for r in recs))}" if recs else ""))
        tot = Counter()
        for m in report["modules"].values():
            tot.update(m["by_kind"]); tot["assets"] += m["assets"]
        report["totals"] = dict(tot)
        (out_dir / "_validation.json").write_text(json.dumps(report, indent=1), encoding="utf-8")
    print("done; re-run sections 1 to 7 to score the new runs")
else:
    print(f"RUN_API is False: nothing generated. Set it to True to run {VERSION_TO_RUN} ({REPEATS} runs, 18 calls each).")

RUN_API is False: nothing generated. Set it to True to run v2p3 (3 runs, 18 calls each).
